# AIM+ FY2026 — Ekstraksi Semua Tabel dari Dashboard
**Wahana Visi Indonesia | PEARL/MEAL**

Script ini membaca semua tabel dari sheet **IND6 Dashboard** dan **IND7 Dashboard**  
(bisa diperluas ke indikator lain dengan menambahkan nama sheet di `INDICATORS`).

**Output yang dihasilkan:**
- 15 tabel × 2 indikator = **30 DataFrame** di notebook
- 1 file Excel (`Output_Semua_Tabel.xlsx`) dengan 30 sheet

---
**Cara pakai:**  
1. Pastikan file `Data AIM+.xlsx` ada di folder yang sama dengan notebook ini
2. Jalankan semua sel (`Run All`)
3. Download file output Excel yang dihasilkan

In [ ]:
# ─── Install jika belum ada ───────────────────────────────────────────────
import subprocess, sys
for pkg in ['openpyxl', 'pandas', 'IPython']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        
print('✓ Semua library siap')

In [ ]:
import openpyxl
import pandas as pd
from collections import OrderedDict
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# ─── KONFIGURASI — ubah di sini ──────────────────────────────────────────
FILE_EXCEL   = 'Data AIM+.xlsx'      # Nama file Excel dashboard
INDICATORS   = ['IND6', 'IND7']      # Tambahkan indikator lain jika mau
OUTPUT_FILE  = 'Output_Semua_Tabel.xlsx'
# ──────────────────────────────────────────────────────────────────────────

print(f'📂 Membuka: {FILE_EXCEL}')
wb = openpyxl.load_workbook(FILE_EXCEL, data_only=True)
print(f'   Sheet tersedia: {wb.sheetnames}')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# FUNGSI INTI: Ekstraksi dinamis semua tabel dari 1 sheet indikator
# ═════════════════════════════════════════════════════════════════════════

def extract_all_tables(ws):
    """
    Membaca semua tabel dari satu sheet indikator dashboard AIM+.
    Mendeteksi section header secara dinamis — tidak hardcode nomor baris.
    
    Returns
    -------
    OrderedDict : {nama_section: DataFrame}
    """
    tables  = OrderedDict()
    max_r   = ws.max_row
    max_c   = ws.max_column

    # ── Temukan semua section header ─────────────────────────────────────
    section_starts = []   # (baris_header_seksi, nama_seksi, baris_kolom_header)

    # Section 1 — ADP Status (header langsung di baris 5, sebelum ada label seksi)
    for r in range(1, 10):
        a = ws.cell(r, 1).value
        b = ws.cell(r, 2).value
        if a == 'ADP' and b == 'Denominator':
            # Section header-nya ada 2 baris sebelumnya
            sec_label = ws.cell(r-2, 1).value or '1. ADP Status'
            section_starts.append((r-2, str(sec_label), r))
            break

    # Section 2 dst — cari label seksi, lalu cari baris kolom header
    for r in range(4, max_r + 1):
        a = ws.cell(r, 1).value
        b = ws.cell(r, 2).value
        if not (a and b is None and isinstance(a, str) and len(str(a)) > 5):
            continue
        # Cari baris kolom header di baris berikutnya (maks 3 baris ke bawah)
        for nr in range(r+1, min(r+4, max_r+1)):
            na = ws.cell(nr, 1).value
            nb = ws.cell(nr, 2).value
            is_header = (
                na in ('ADP', 'Category') or
                (nb and isinstance(nb, str) and
                 any(k in str(nb) for k in ['Denom','Weight','HH','Rate','Weighted']))
            )
            if is_header:
                section_starts.append((r, str(a), nr))
                break
        if r > 295:
            break

    # ── Baca data tiap seksi ──────────────────────────────────────────────
    def read_block(header_row, stop_before):
        """Baca dari header_row+1 sampai col-A kosong atau baris stop_before."""
        # Ambil nama kolom
        headers = []
        for c in range(1, max_c + 1):
            v = ws.cell(header_row, c).value
            if v is None and c > 3:
                break
            headers.append(str(v) if v is not None else f'Kolom_{c}')
        while headers and headers[-1].startswith('Kolom_'):
            headers.pop()

        # Baca baris data
        rows = []
        r = header_row + 1
        while r <= max_r:
            if stop_before and r >= stop_before:
                break
            cell_a = ws.cell(r, 1).value
            if cell_a is None or cell_a == '':
                break
            row = [ws.cell(r, c).value for c in range(1, len(headers)+1)]
            rows.append(row)
            r += 1

        df = pd.DataFrame(rows, columns=headers).dropna(how='all')
        return df

    for i, (sec_r, sec_name, hdr_r) in enumerate(section_starts):
        next_stop = section_starts[i+1][0] if i+1 < len(section_starts) else max_r+1
        df = read_block(hdr_r, stop_before=next_stop)
        tables[sec_name] = df

    return tables


# ── Jalankan untuk semua indikator ───────────────────────────────────────
all_results = {}
for ind in INDICATORS:
    sheet_name = f'{ind} Dashboard'
    if sheet_name not in wb.sheetnames:
        print(f'⚠️  Sheet "{sheet_name}" tidak ditemukan, dilewati.')
        continue
    ws = wb[sheet_name]
    all_results[ind] = extract_all_tables(ws)
    n = len(all_results[ind])
    print(f'✓ {ind}: {n} tabel diekstrak')

total = sum(len(v) for v in all_results.values())
print(f'\n📊 Total: {total} tabel dari {len(all_results)} indikator')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# TAMPILKAN SEMUA TABEL DI NOTEBOOK
# ═════════════════════════════════════════════════════════════════════════

# Styling tabel HTML
TABLE_CSS = """
<style>
.aim-table { border-collapse: collapse; font-family: Arial, sans-serif;
             font-size: 11px; margin-bottom: 8px; width: 100%; }
.aim-table th { background: #003A6B; color: white; padding: 6px 10px;
                text-align: center; border: 1px solid #ccc; }
.aim-table td { padding: 5px 10px; border: 1px solid #ddd; text-align: center; }
.aim-table tr:nth-child(even) { background: #F0F6FF; }
.aim-table tr:hover { background: #D6E8FF; }
.aim-table td:first-child { text-align: left; font-weight: 500; }
.sec-title { font-family: Arial; font-size: 13px; font-weight: bold;
             color: #003A6B; margin: 20px 0 4px 0; 
             border-left: 4px solid #F37021; padding-left: 8px; }
.ind-title { font-family: Arial; font-size: 16px; font-weight: bold;
             color: white; background: #003A6B; padding: 8px 14px;
             margin-top: 28px; border-radius: 4px; }
.shape-badge { font-size: 10px; color: #666; margin-left: 8px; 
               font-weight: normal; }
</style>
"""

def format_cell(v):
    """Format nilai sel: rate desimal → persen, '-' tetap '-'."""
    if v is None or v == '-':  return '-'
    if isinstance(v, float):
        if 0 < abs(v) < 1 and 'Rate' not in str(v):   # Weight/Rate desimal
            return f'{v:.2%}'
        return f'{v:.4f}' if abs(v) < 0.01 else f'{v:.2f}'
    return str(v)

def df_to_html(df, max_rows=20):
    """Render DataFrame sebagai tabel HTML dengan styling."""
    rows_html = ''
    for _, row in df.head(max_rows).iterrows():
        cells = ''.join(f'<td>{format_cell(v)}</td>' for v in row)
        rows_html += f'<tr>{cells}</tr>'
    if len(df) > max_rows:
        rows_html += f'<tr><td colspan="{len(df.columns)}" style="color:#999;font-style:italic;">... +{len(df)-max_rows} baris lagi</td></tr>'
    headers = ''.join(f'<th>{c}</th>' for c in df.columns)
    return f'<table class="aim-table"><thead><tr>{headers}</tr></thead><tbody>{rows_html}</tbody></table>'


# ── Render ────────────────────────────────────────────────────────────────
html_out = TABLE_CSS
for ind, tables in all_results.items():
    html_out += f'<div class="ind-title">{ind} Dashboard — {len(tables)} tabel</div>'
    for sec_name, df in tables.items():
        badge = f'<span class="shape-badge">[{df.shape[0]} baris × {df.shape[1]} kolom]</span>'
        html_out += f'<div class="sec-title">{sec_name}{badge}</div>'
        html_out += df_to_html(df)

display(HTML(html_out))

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# EKSPOR KE EXCEL — 1 sheet per tabel
# ═════════════════════════════════════════════════════════════════════════

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

WVI_BLUE   = 'FF003A6B'
WVI_ORANGE = 'FFF37021'
WVI_LIGHT  = 'FFD6E8FF'
WHITE      = 'FFFFFFFF'
GREY       = 'FFF5F5F5'

thin = Side(style='thin', color='FFD0D0D0')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def write_sheet(ws_out, df, title, subtitle):
    """Tulis satu DataFrame ke satu sheet Excel dengan styling WVI."""
    max_col = len(df.columns)
    col_letter = get_column_letter(max_col)

    # Title row
    ws_out.merge_cells(f'A1:{col_letter}1')
    c = ws_out['A1']
    c.value = title
    c.font = Font(bold=True, color=WHITE, size=12, name='Arial')
    c.fill = PatternFill('solid', start_color=WVI_BLUE, end_color=WVI_BLUE)
    c.alignment = Alignment(horizontal='left', vertical='center')
    ws_out.row_dimensions[1].height = 22

    # Subtitle row
    ws_out.merge_cells(f'A2:{col_letter}2')
    c2 = ws_out['A2']
    c2.value = subtitle
    c2.font = Font(italic=True, color='FF555555', size=9, name='Arial')
    c2.fill = PatternFill('solid', start_color=GREY, end_color=GREY)
    c2.alignment = Alignment(horizontal='left', vertical='center')
    ws_out.row_dimensions[2].height = 16

    # Column headers row 3
    for ci, col_name in enumerate(df.columns, 1):
        c = ws_out.cell(3, ci, value=col_name)
        c.font = Font(bold=True, color=WHITE, size=9, name='Arial')
        c.fill = PatternFill('solid', start_color=WVI_ORANGE, end_color=WVI_ORANGE)
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = border
    ws_out.row_dimensions[3].height = 28

    # Data rows
    for ri, (_, row) in enumerate(df.iterrows(), 4):
        bg = WVI_LIGHT if ri % 2 == 0 else WHITE
        for ci, val in enumerate(row, 1):
            # Format rate desimal → persen
            display_val = val
            if isinstance(val, float) and 0 < abs(val) < 1:
                display_val = round(val * 100, 2)
            c = ws_out.cell(ri, ci, value=display_val)
            c.font = Font(size=9, name='Arial')
            c.fill = PatternFill('solid', start_color=bg, end_color=bg)
            c.alignment = Alignment(
                horizontal='left' if ci == 1 else 'center',
                vertical='center'
            )
            c.border = border

    # Auto column width
    for ci, col_name in enumerate(df.columns, 1):
        max_len = max(
            len(str(col_name)),
            df.iloc[:, ci-1].astype(str).str.len().max() if len(df) > 0 else 8
        )
        ws_out.column_dimensions[get_column_letter(ci)].width = min(max_len + 3, 35)


# ── Buat workbook output ─────────────────────────────────────────────────
wb_out = Workbook()
wb_out.remove(wb_out.active)

# Index sheet — daftar semua sheet
ws_idx = wb_out.create_sheet('📋 Daftar Isi')
ws_idx.column_dimensions['A'].width = 8
ws_idx.column_dimensions['B'].width = 20
ws_idx.column_dimensions['C'].width = 45
ws_idx.column_dimensions['D'].width = 20
ws_idx.column_dimensions['E'].width = 12

idx_header = ['No', 'Indikator', 'Nama Tabel', 'Nama Sheet', 'Ukuran (baris×kolom)']
for ci, h in enumerate(idx_header, 1):
    c = ws_idx.cell(1, ci, value=h)
    c.font = Font(bold=True, color=WHITE, size=10, name='Arial')
    c.fill = PatternFill('solid', start_color=WVI_BLUE, end_color=WVI_BLUE)
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border = border
ws_idx.row_dimensions[1].height = 24

idx_row = 2
sheet_no = 0

for ind, tables in all_results.items():
    for sec_name, df in tables.items():
        sheet_no += 1

        # Buat nama sheet singkat (maks 31 karakter)
        sec_short = sec_name[:25].strip()
        sheet_label = f'{ind}_{sheet_no:02d}'

        # Buat sheet
        ws_new = wb_out.create_sheet(sheet_label)
        ws_new.sheet_view.showGridLines = False
        ws_new.freeze_panes = 'B4'
        title    = f'AIM+ FY2026 | {ind} Dashboard'
        subtitle = f'Tabel {sheet_no}: {sec_name}   [{df.shape[0]} baris × {df.shape[1]} kolom]'
        write_sheet(ws_new, df, title, subtitle)

        # Tulis ke index
        row_vals = [sheet_no, ind, sec_name, sheet_label, f'{df.shape[0]}×{df.shape[1]}']
        for ci, v in enumerate(row_vals, 1):
            c = ws_idx.cell(idx_row, ci, value=v)
            c.font = Font(size=9, name='Arial')
            c.fill = PatternFill('solid', start_color=WVI_LIGHT if idx_row%2==0 else WHITE,
                                 end_color=WVI_LIGHT if idx_row%2==0 else WHITE)
            c.alignment = Alignment(horizontal='left', vertical='center')
            c.border = border
        idx_row += 1

wb_out.save(OUTPUT_FILE)
print(f'✅ File disimpan: {OUTPUT_FILE}')
print(f'   Total sheet: {len(wb_out.sheetnames)-1} tabel + 1 daftar isi')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# BONUS: Akses tabel spesifik langsung sebagai DataFrame
# ═════════════════════════════════════════════════════════════════════════

print('=== Daftar semua tabel yang tersedia ===')
for ind, tables in all_results.items():
    print(f'\n{ind}:')
    for name in tables:
        print(f'  all_results["{ind}"]["{name}"]')

In [ ]:
# Contoh: ambil tabel tertentu langsung
# Ubah nama di bawah sesuai kebutuhan

# Contoh 1 — ADP Status IND6
df_ind6_s1 = all_results['IND6']['1. ADP Status']
print('=== IND6: ADP Status (Section 1) ===')
display(df_ind6_s1)

# Contoh 2 — Direct vs Indirect IND7
df_ind7_di = all_results['IND7']['7. ADP Status by Direct/Indirect Beneficiary']
print('\n=== IND7: Direct/Indirect (Section 7) ===')
display(df_ind7_di)

# Contoh 3 — Bandingkan rate IND6 vs IND7 per AP
s1_6 = all_results['IND6']['1. ADP Status'][['ADP','Rate (%)']].rename(columns={'Rate (%)':'Rate IND6'})
s1_7 = all_results['IND7']['1. ADP Status'][['ADP','Rate (%)']].rename(columns={'Rate (%)':'Rate IND7'})
compare = s1_6.merge(s1_7, on='ADP')
compare['Rate IND6 (%)'] = compare['Rate IND6'].apply(lambda x: f"{x*100:.2f}%" if isinstance(x,(int,float)) else x)
compare['Rate IND7 (%)'] = compare['Rate IND7'].apply(lambda x: f"{x*100:.2f}%" if isinstance(x,(int,float)) else x)
print('\n=== Perbandingan Rate IND6 vs IND7 per AP ===')
display(compare[['ADP','Rate IND6 (%)','Rate IND7 (%)']])